<a href="https://colab.research.google.com/github/hbprosper/dihiggs/blob/main/code/heft_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Higgs Effective Field Theory (HEFT) Study: Export
> Created: Oct 26, 2026 Pushpa Bhat (Fermilab), Nicola de Filippis (Bari),<br> Kurtis Johnson,  Harrison B. Prosper, Ryne Starnes (FSU)<br>

## Introduction

This notebook makes the trained model available to colleagues who do not use PyTorch. It writes two self-contained copies of the model:

| folder | the model | needs |
| :--- | :--- | :--- |
| `exported/python/` | `heftnet_numpy.py` | numpy and the Python standard library |
| `exported/cpp/` | `heftnet.hpp` (header only) | the C++17 standard library |

Both copies read the trained parameters from the same plain-text file, `heftnet_weights.txt`, and take the di-Higgs mass in **GeV** (the PyTorch model takes the dimensionless variable $x = (m_{hh} - 255\,\text{GeV})/1500\,\text{GeV}$). Each folder also contains `reference.txt`, a set of inputs and outputs of the PyTorch model, and an example program that checks the copy against it.

The work is done by the module `dihiggs.convert`, which also validates both copies against the PyTorch model. The model itself is described in `docs/dihiggs_model.pdf`.

Run `heft_train.ipynb` first: this notebook exports the model it trained.

In [ ]:
# ------------------------------------------------------------------
# Google Colab setup (nothing happens when run outside Colab)
#   1. clone the dihiggs repository, which holds the data files,
#   2. install the dihiggs package, which also installs drwho,
#   3. make dihiggs/code the working directory so that the relative
#      paths used below (../data, runs/) work as they do locally.
# ------------------------------------------------------------------
import os, sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules

# Each Colab notebook runs on its own temporary machine. With USE_DRIVE = True
# the runs/ folder lives in your Google Drive (MyDrive/dihiggs/runs), where
# heft_train.ipynb saved the trained model when run with USE_DRIVE = True.
USE_DRIVE = True

if IN_COLAB:
    REPO = '/content/dihiggs'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/hbprosper/dihiggs.git', REPO],
                       check=True)
    # A regular (non-editable) install can be imported at once; after an
    # editable install (pip install -e) the runtime must be restarted
    # before "import dihiggs" works.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', REPO],
                   check=True)
    importlib.invalidate_caches()
    os.chdir(f'{REPO}/code')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_runs = '/content/drive/MyDrive/dihiggs/runs'
        os.makedirs(drive_runs, exist_ok=True)
        if not os.path.exists('runs'):
            os.symlink(drive_runs, 'runs')
        print(f'runs/ is stored in {drive_runs}')

print(f'working directory: {os.getcwd()}')

In [ ]:
import os, sys, shutil, subprocess

# the standard module for tabular data
import pandas as pd

# the standard module for array manipulation
import numpy as np

# the standard modules for high-quality plots
import matplotlib as mp
import matplotlib.pyplot as plt

# the converter
from dihiggs import convert

# update fonts
plt.rcParams.update({
  "text.usetex": shutil.which('latex') is not None,
  "font.family": "sans-serif",
  "font.size": 12
})

## Choose the trained model

The configuration file written by `heft_train.ipynb` records where the trained parameters are.

In [ ]:
DIRNAME = 'L1Loss'
NAME    = 'dihiggs'
BASE    = f'runs/{DIRNAME}'

config_file = f'{BASE}/{NAME}_config.yaml'
if not os.path.exists(config_file):
    raise FileNotFoundError(
        f'{config_file} not found: run heft_train.ipynb first. On Colab, '
        'run both notebooks with USE_DRIVE = True so that they share runs/.')

OUTDIR = 'exported'

## Export and validate

`convert.export` does four things for each language:
 1. copies the model code (`heftnet_numpy.py` or `heftnet.hpp`), an example program and a README into the output folder;
 2. writes the trained parameters to `heftnet_weights.txt`;
 3. writes `reference.txt`: the PyTorch model's prediction for the full spectrum (80 bins) at seven parameter points, including the Standard Model;
 4. validates the copy. The numpy copy is compared with the PyTorch model, computed in double precision, at every bin of the 500 held-out test spectra; the C++ copy is compiled (if a C++ compiler is available) and its example program checks it against `reference.txt`.

The parameters are written with enough digits to be read back exactly, so the copies agree with PyTorch to about $10^{-11}$, limited only by the rounding of the arithmetic.

In [ ]:
folders = convert.export(config_file, outdir=OUTDIR)

## The weights file

The file is plain text, so it can be read with any editor or parsed in any language. The header describes the format; each tensor is introduced by a line `tensor NAME NDIM D1 ... DNDIM`.

In [ ]:
with open(f'{OUTDIR}/python/heftnet_weights.txt') as f:
    lines = f.readlines()
print(''.join(lines[:32]))
print(f'... {len(lines)} lines in all')

## Use the numpy copy

Import `heftnet_numpy.py` from the export folder, exactly as a colleague would, and compare it with the PyTorch model on the held-out test spectra.

In [ ]:
sys.path.insert(0, f'{OUTDIR}/python')
from heftnet_numpy import HEFTNet

model = HEFTNet(f'{OUTDIR}/python/heftnet_weights.txt')

# all bins of all test spectra
mhh, theta, description = convert.test_points(config_file)
print(description)

sigma_numpy = model(mhh, theta)

# the PyTorch model, in double precision
trained = convert.load_trained(config_file)
sigma_torch = convert.torch_sigma(trained['model'], mhh, theta)

error = convert.relative_error(sigma_numpy, sigma_torch)
print(f'largest relative difference: {error.max():.2e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.loglog(sigma_torch, sigma_numpy, '.', markersize=2)
lim = [sigma_torch[sigma_torch > 0].min(), sigma_torch.max()]
ax.plot(lim, lim, 'r-', linewidth=1)
ax.set_xlabel(r'$\sigma$ (PyTorch) [pb]')
ax.set_ylabel(r'$\sigma$ (numpy copy) [pb]')
ax.grid(True, which='both', alpha=0.3)

ax = axes[1]
exact = error == 0                         # bit-for-bit agreement
ax.hist(np.log10(error[~exact]), bins=60)
ax.set_title(f'{exact.sum()} of {len(error)} points agree exactly (not shown)',
             fontsize=10)
ax.set_xlabel(r'$\log_{10}$ relative difference')
ax.set_ylabel('number of points')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### A spectrum

The copy takes the mass in GeV. The training range is $255 \le m_{hh} \le 1455$ GeV; outside it the copy warns that the model is extrapolating.

In [ ]:
SM = [1, 1, 0, 0, 0]                      # klambda, ct, ctt, cggh, cgghh
centres = np.arange(262.5, 1455, 15)       # the 80 bin centres in GeV
edges = np.arange(255, 1456, 15)

fig, ax = plt.subplots(figsize=(6, 4))
for theta_point, label in [(SM, 'SM'), ([5, 1, 0, 0, 0], r'$\kappa_\lambda = 5$')]:
    ax.stairs(model(centres, theta_point), edges, label=label, linewidth=1.5,
              baseline=None)
ax.set_yscale('log')
ax.set_xlabel(r'$m_{hh}$ [GeV]')
ax.set_ylabel(r'$\sigma$ per 15 GeV bin [pb]')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Use the C++ copy

Compile and run the example program, which prints the Standard Model spectrum and checks the copy against `reference.txt`.

In [ ]:
compiler = shutil.which('c++') or shutil.which('g++') or shutil.which('clang++')
if compiler is None:
    print('no C++ compiler found')
else:
    cpp = f'{OUTDIR}/cpp'
    subprocess.run([compiler, '-std=c++17', '-O2', '-o', 'example', 'example.cpp'],
                   cwd=cpp, check=True)
    run = subprocess.run(['./example'], cwd=cpp, capture_output=True, text=True)
    print(run.stdout)
    os.remove(f'{cpp}/example')

## Package for colleagues

Each folder is self-contained. Zip them for sharing. On Colab, download the zip files from the file browser on the left.

In [ ]:
for language in ['python', 'cpp']:
    archive = shutil.make_archive(f'{OUTDIR}/dihiggs_heftnet_{language}', 'zip',
                                  root_dir=OUTDIR, base_dir=language)
    print(f'wrote {archive}')